In [ ]:
from pathlib import Path
import json

from PIL import Image, ImageOps
from torchvision import transforms

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

split_manifest = json.loads(
    (PROJECT_ROOT / "reports" / "base_split.json").read_text(
        encoding="utf-8"
    )
)
print("Sources:", split_manifest["sources"])
print("Split:", len(split_manifest["training_paths"]), len(split_manifest["validation_paths"]))

IMAGE_SIZE = 128

def resize_with_padding(image):
    return ImageOps.pad(
        image.convert("RGB"),
        (IMAGE_SIZE, IMAGE_SIZE),
        method=Image.Resampling.BILINEAR,
        color=(0, 0, 0),
    )

base_transform = transforms.Compose([
    transforms.Lambda(resize_with_padding),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5],
    ),
])

sample_path = PROJECT_ROOT / split_manifest["training_paths"][0]

with Image.open(sample_path) as image:
    sample_tensor = base_transform(image)

print("Input tensor shape:", sample_tensor.shape)
print("Pixel range:", sample_tensor.min().item(), sample_tensor.max().item())

In [ ]:
with Image.open(sample_path) as image:
    prepared = resize_with_padding(image)

    print("Original:", image.size)
    print("Prepared:", prepared.size)
    display(prepared)

In [ ]:
from torch.utils.data import Dataset

class ManifestImageDataset(Dataset):
    """One class mapping and eligible sample list across all source folders."""
    def __init__(self, manifest, transform=None):
        self.transform = transform
        self.class_to_idx = manifest["class_to_idx"]
        self.classes = sorted(self.class_to_idx, key=self.class_to_idx.get)
        self.samples = [
            (str(PROJECT_ROOT / row["path"]), self.class_to_idx[row["label"]])
            for row in manifest["samples"]
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        with Image.open(path) as image:
            image = image.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label

train_dataset = ManifestImageDataset(split_manifest)
print("Eligible images:", len(train_dataset))
print("Class mapping:", train_dataset.class_to_idx)

In [ ]:
path_to_index = {
    Path(path).relative_to(PROJECT_ROOT).as_posix(): index
    for index, (path, label) in enumerate(train_dataset.samples)
}
train_indices = [path_to_index[path] for path in split_manifest["training_paths"]]
val_indices = [path_to_index[path] for path in split_manifest["validation_paths"]]
assert train_indices and val_indices
assert not set(train_indices) & set(val_indices)
assert set(train_indices) | set(val_indices) == set(range(len(train_dataset)))
print("Training indices:", len(train_indices))
print("Validation indices:", len(val_indices))

In [ ]:
from torch.utils.data import Subset

# Separate transforms can be applied later to this same eligible sample list.
train_full = ManifestImageDataset(split_manifest, transform=base_transform)
validation_full = ManifestImageDataset(split_manifest, transform=base_transform)
assert train_dataset.samples == train_full.samples == validation_full.samples
train_subset = Subset(train_full, train_indices)
validation_subset = Subset(validation_full, val_indices)
print("Training images:", len(train_subset))
print("Validation images:", len(validation_subset))
print("Classes:", train_full.classes)
image_tensor, label_index = train_subset[0]
print("One image shape:", image_tensor.shape)
print("Its class:", train_full.classes[label_index])

In [ ]:
import torch
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(42),
)

validation_loader = DataLoader(
    validation_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))

batch_images, batch_labels = next(iter(train_loader))

print("Images shape:", batch_images.shape)
print("Labels shape:", batch_labels.shape)
print("First five labels:", batch_labels[:5])


In [ ]:
from torch import nn

# Conv2d output per spatial axis (dilation=1): floor((N + 2*P - K) / S) + 1.
# Here N=128, P=1, K=3, S=1: floor((128 + 2*1 - 3) / 1) + 1 = 128.
# 16 filters make 16 feature maps of 128x128; MaxPool2d(2) makes them 64x64.
first_block = nn.Sequential(
    nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
)

first_output = first_block(batch_images)

print("Input shape:", batch_images.shape)
print("Output shape:", first_output.shape)

In [ ]:
# Here N=64, P=1, K=3, S=1: floor((64 + 2*1 - 3) / 1) + 1 = 64.
# 32 filters make 32 feature maps of 64x64; MaxPool2d(2) makes them 32x32.
second_block = nn.Sequential(
    nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
)

second_output = second_block(first_output)

print("First block output:", first_output.shape)
print("Second block output:", second_output.shape)

In [ ]:
flatten = nn.Flatten(start_dim=1)
flat_features = flatten(second_output)

print("Before flatten:", second_output.shape)
print("After flatten:", flat_features.shape)

In [ ]:
num_classes = len(train_full.classes)

classifier = nn.Linear(
    in_features=32 * 32 * 32,
    out_features=num_classes,
)

class_scores = classifier(flat_features)

print("Number of classes:", num_classes)
print("Scores shape:", class_scores.shape)
print("First image scores:", class_scores[0])

In [ ]:
first_image_scores = class_scores[0]

for class_index, score in enumerate(first_image_scores):
    class_name = train_full.classes[class_index]
    print(f"{class_index}: {class_name:10s} | score: {score.item():.4f}")

predicted_index = first_image_scores.argmax().item()
target_index = batch_labels[0].item()

print("Predicted class:", train_full.classes[predicted_index])
print("True class:", train_full.classes[target_index])

In [ ]:
loss_fn = nn.CrossEntropyLoss()
batch_loss = loss_fn(class_scores, batch_labels)

per_image_losses = nn.functional.cross_entropy(
    class_scores,
    batch_labels,
    reduction="none",
)

print("Scores shape:", class_scores.shape)
print("Labels shape:", batch_labels.shape)
print("First image loss:", per_image_losses[0].item())
print("Mean of image losses:", per_image_losses.mean().item())
print("Batch loss:", batch_loss.item())

In [ ]:
layers = {
    "first conv": first_block[0],
    "second conv": second_block[0],
    "classifier": classifier,
}

for name, layer in layers.items():
    print(name, "gradient before backward:", layer.weight.grad is None)

batch_loss.backward()

for name, layer in layers.items():
    print(
        name,
        "gradient shape:", tuple(layer.weight.grad.shape),
        "mean absolute gradient:", layer.weight.grad.abs().mean().item(),
    )

In [ ]:
all_parameters = [
    *first_block.parameters(),
    *second_block.parameters(),
    *classifier.parameters(),
]

optimizer = torch.optim.SGD(all_parameters, lr=0.01)

old_loss = batch_loss.item()
old_classifier_weights = classifier.weight.detach().clone()

optimizer.step()

with torch.no_grad():
    new_scores = classifier(
        flatten(
            second_block(
                first_block(batch_images)
            )
        )
    )
    new_loss = loss_fn(new_scores, batch_labels)

largest_weight_change = (
    classifier.weight.detach() - old_classifier_weights
).abs().max().item()

print("Old loss:", old_loss)
print("New loss:", new_loss.item())
print("Largest classifier weight change:", largest_weight_change)

In [ ]:
print("Before clearing:", classifier.weight.grad is None)
print("Gradient size:", classifier.weight.grad.abs().mean().item())

optimizer.zero_grad(set_to_none=True)

for name, layer in layers.items():
    print(name, "gradient after clearing:", layer.weight.grad)

In [ ]:
class VehicleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Linear(32 * 32 * 32, num_classes),
        )

    def forward(self, images):
        features = self.features(images)
        scores = self.classifier(features)
        return scores


torch.manual_seed(42)
model = VehicleCNN(num_classes=len(train_full.classes))

print(model)
print("Output shape:", model(batch_images).shape)
print("Parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model = model.to(device)

LEARNING_RATE = 1e-3

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

print("Device:", device)
print("Learning rate:", LEARNING_RATE)
print("Optimizer:", type(optimizer).__name__)

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()

    total_loss = 0.0
    correct_predictions = 0
    total_images = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        scores = model(images)
        loss = loss_fn(scores, labels)

        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size

        predicted_indices = scores.argmax(dim=1)
        correct_predictions += (
            predicted_indices == labels
        ).sum().item()

        total_images += batch_size

    return {
        "loss": total_loss / total_images,
        "accuracy": correct_predictions / total_images,
    }

In [ ]:
def evaluate(model, loader, loss_fn, device):
    model.eval()

    total_loss = 0.0
    correct_predictions = 0
    total_images = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            scores = model(images)
            loss = loss_fn(scores, labels)

            batch_size = images.size(0)
            total_loss += loss.item() * batch_size

            predicted_indices = scores.argmax(dim=1)
            correct_predictions += (
                predicted_indices == labels
            ).sum().item()

            total_images += batch_size

    return {
        "loss": total_loss / total_images,
        "accuracy": correct_predictions / total_images,
    }


In [ ]:
initial_validation = evaluate(
    model,
    validation_loader,
    loss_fn,
    device,
)

print("Initial validation loss:", initial_validation["loss"])
print(
    "Initial validation accuracy:",
    f'{initial_validation["accuracy"]:.1%}',
)

In [ ]:
from copy import deepcopy

# Start this experiment with fresh weights and a fixed shuffle seed.
torch.manual_seed(42)
train_loader.generator.manual_seed(42)

model = VehicleCNN(
    num_classes=len(train_full.classes)
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

EPOCHS = 10
history = []

best_val_accuracy = -1.0
best_epoch = 0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_metrics = train_one_epoch(
        model, train_loader, optimizer, loss_fn, device
    )
    val_metrics = evaluate(
        model, validation_loader, loss_fn, device
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
    })

    if val_metrics["accuracy"] > best_val_accuracy:
        best_val_accuracy = val_metrics["accuracy"]
        best_epoch = epoch
        best_state = deepcopy(model.state_dict())

    print(
        f'Epoch {epoch:02d}/{EPOCHS} | '
        f'train loss: {train_metrics["loss"]:.4f} | '
        f'train acc: {train_metrics["accuracy"]:.1%} | '
        f'val loss: {val_metrics["loss"]:.4f} | '
        f'val acc: {val_metrics["accuracy"]:.1%}'
    )

print(
    f"Best validation accuracy: {best_val_accuracy:.1%} "
    f"at epoch {best_epoch}"
)

## تفسیر نتیجهٔ CNN پایه

پس از اجرای سلول آموزش، از `history` بهترین epoch، دقت و loss آموزش و validation را بخوان. فاصلهٔ زیاد دقت آموزش و validation می‌تواند نشانهٔ بیش‌برازش باشد. عددهای اجرای قبلی به تقسیم قبلی مربوط‌اند؛ این بخش پس از آموزش روی تقسیم یکپارچه باید با خروجی تازه تکمیل شود.


In [ ]:
checkpoint_dir = PROJECT_ROOT / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = checkpoint_dir / "cnn_baseline_best.pt"

checkpoint = {
    "architecture": "VehicleCNN",
    "model_state_dict": {
        name: tensor.detach().cpu()
        for name, tensor in best_state.items()
    },
    "class_to_idx": train_full.class_to_idx,
    "best_epoch": best_epoch,
    "best_val_accuracy": best_val_accuracy,
    "preprocessing": {
        "image_size": IMAGE_SIZE,
        "resize": "aspect-ratio-preserving padding",
        "interpolation": "bilinear",
        "padding_color": [0, 0, 0],
        "mean": [0.5, 0.5, 0.5],
        "std": [0.5, 0.5, 0.5],
    },
    "training_config": {
        "seed": 42,
        "optimizer": "Adam",
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "loss": "CrossEntropyLoss",
    },
    "split_fingerprint": split_manifest["fingerprint"],
    "history": history,
    "review_threshold": None,  # Not calibrated yet.
}

# This is a binary file; do not edit or resave it as text.
torch.save(checkpoint, checkpoint_path)

# Verify that the saved checkpoint can be read immediately.
verified_checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)
assert verified_checkpoint["best_epoch"] == best_epoch
assert verified_checkpoint["class_to_idx"] == train_full.class_to_idx
print("Checkpoint read-back verified.")

print("Saved to:", checkpoint_path)
print("Best epoch:", checkpoint["best_epoch"])
print("Validation accuracy:", f'{best_val_accuracy:.1%}')

In [ ]:
import matplotlib.pyplot as plt

epochs = [row["epoch"] for row in history]
train_losses = [row["train_loss"] for row in history]
val_losses = [row["val_loss"] for row in history]

train_accuracies = [
    100 * row["train_accuracy"] for row in history
]
val_accuracies = [
    100 * row["val_accuracy"] for row in history
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, train_losses, marker="o", label="Train")
axes[0].plot(epochs, val_losses, marker="o", label="Validation")
axes[0].set_title("Cross-entropy loss")
axes[0].set_ylabel("Loss")

axes[1].plot(epochs, train_accuracies, marker="o", label="Train")
axes[1].plot(epochs, val_accuracies, marker="o", label="Validation")
axes[1].set_title("Classification accuracy")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_ylim(0, 100)

for ax in axes:
    ax.set_xlabel("Epoch")
    ax.set_xticks(epochs)
    ax.axvline(
        best_epoch,
        linestyle="--",
        color="gray",
        label="Selected epoch",
    )
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()

figure_dir = PROJECT_ROOT / "reports" / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(
    figure_dir / "cnn_baseline_curves.png",
    dpi=150,
    bbox_inches="tight",
)

plt.show()

In [ ]:
checkpoint_path = PROJECT_ROOT / "checkpoints" / "cnn_baseline_best.pt"

# Recover a checkpoint accidentally resaved as UTF-16 text.
with checkpoint_path.open("rb") as checkpoint_file:
    header = checkpoint_file.read(2)

if header in (b"\xfe\xff", b"\xff\xfe"):
    if "checkpoint" not in globals():
        raise RuntimeError(
            "The checkpoint was converted to text. "
            "Run the checkpoint save cell using best_state still in memory."
        )
    assert checkpoint["class_to_idx"] == train_full.class_to_idx
    torch.save(checkpoint, checkpoint_path)
    print("Restored binary checkpoint from the in-memory checkpoint dictionary.")

saved_checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)

assert (
    saved_checkpoint["class_to_idx"]
    == train_full.class_to_idx
)

# Current checkpoints store only the fingerprint; accept older files for the check.
saved_split_fingerprint = saved_checkpoint.get("split_fingerprint")
if saved_split_fingerprint is None:
    saved_split_fingerprint = saved_checkpoint.get("split_manifest", {}).get("fingerprint")
if saved_split_fingerprint != split_manifest["fingerprint"]:
    raise RuntimeError("Checkpoint belongs to a different data split. Train and save with the current split first.")

best_model = VehicleCNN(
    num_classes=len(saved_checkpoint["class_to_idx"])
).to(device)

best_model.load_state_dict(
    saved_checkpoint["model_state_dict"]
)

best_validation = evaluate(
    best_model,
    validation_loader,
    loss_fn,
    device,
)

print("Loaded epoch:", saved_checkpoint["best_epoch"])
print("Validation loss:", best_validation["loss"])
print("Validation accuracy:", f'{best_validation["accuracy"]:.1%}')

In [ ]:
best_model.eval()

y_true = []
y_pred = []
probability_batches = []

with torch.no_grad():
    for images, labels in validation_loader:
        scores = best_model(images.to(device))
        probabilities = torch.softmax(scores, dim=1)
        predictions = scores.argmax(dim=1)

        y_true.extend(labels.tolist())
        y_pred.extend(predictions.cpu().tolist())
        probability_batches.append(probabilities.cpu())

val_probabilities = torch.cat(probability_batches, dim=0)

correct = sum(
    true_label == predicted_label
    for true_label, predicted_label in zip(y_true, y_pred)
)

print("Number of images:", len(y_true))
print("Probability shape:", val_probabilities.shape)
print("Correct predictions:", correct)
print("First ten true labels:", y_true[:10])
print("First ten predictions:", y_pred[:10])

In [ ]:
from sklearn.metrics import classification_report

class_names = train_full.classes
class_indices = list(range(len(class_names)))
report_text = classification_report(
    y_true, y_pred, labels=class_indices, target_names=class_names,
    digits=3, zero_division=0,
)
print(report_text)

baseline_report = classification_report(
    y_true, y_pred, labels=class_indices, target_names=class_names,
    output_dict=True, zero_division=0,
)
baseline_report.update({
    "experiment": "cnn_baseline",
    "split_fingerprint": split_manifest["fingerprint"],
    "result_status": "current_split",
    "training_images": len(train_subset),
    "validation_images": len(validation_subset),
    "best_epoch": best_epoch,
})
report_path = PROJECT_ROOT / "reports/experiment_results.json"
all_results = json.loads(report_path.read_text()) if report_path.exists() else {}
all_results["_split"] = {
    "current_fingerprint": split_manifest["fingerprint"],
    "comparison_rule": "Compare results only when split_fingerprint matches.",
}
all_results["cnn_baseline"] = baseline_report
report_path.write_text(json.dumps(all_results, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


## تحلیل هر کلاس روی validation فعلی

گزارش سلول قبل برای هر کلاس `precision`، `recall`، `F1` و تعداد نمونه‌های واقعی (`support`) را نشان می‌دهد. کمترین precision یعنی پیش‌بینی‌های مثبتِ اشتباه بیشتر است؛ کمترین recall یعنی نمونه‌های واقعی بیشتری از دست رفته‌اند. برای نتیجه‌گیری دربارهٔ علت خطا، ماتریس خطا و خود تصاویر را هم بررسی کن. بخشی از برچسب‌های `unclean` در این validation هنوز موقت‌اند.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm_counts = confusion_matrix(
    y_true, y_pred, labels=class_indices
)

cm_normalized = confusion_matrix(
    y_true, y_pred, labels=class_indices, normalize="true"
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ConfusionMatrixDisplay(
    cm_counts, display_labels=class_names
).plot(
    ax=axes[0],
    cmap="Blues",
    values_format="d",
    xticks_rotation=45,
    colorbar=False,
)

ConfusionMatrixDisplay(
    cm_normalized, display_labels=class_names
).plot(
    ax=axes[1],
    cmap="Blues",
    values_format=".2f",
    xticks_rotation=45,
    colorbar=False,
)

axes[0].set_title("Validation — image counts")
axes[1].set_title("Validation — normalized by true class")

plt.tight_layout()
plt.savefig(
    figure_dir / "cnn_baseline_confusion_matrices.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## ماتریس خطای جداگانه برای هر کلاس

برای هر کلاس، مسئله را موقتاً به «این کلاس» و «سایر کلاس‌ها» تبدیل می‌کنیم. در هر ماتریس ۲×۲، **سطر برچسب واقعی** و **ستون پیش‌بینی** است:

| | پیش‌بینیِ سایر | پیش‌بینیِ کلاس |
|---|---:|---:|
| واقعیِ سایر | TN | FP |
| واقعیِ کلاس | FN | TP |

برای مثال، تصویری که واقعاً `kamyun` است ولی `kamyunet` پیش‌بینی شده، برای `kamyun` یک FN و برای `kamyunet` یک FP است. ماتریس ۸×۸ قبلی همچنان برای دیدن *جفت کلاس‌های اشتباه‌شده* لازم است؛ این هشت ماتریس برای محاسبهٔ TP/FP/FN/TN هر کلاس‌اند.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for class_index, (class_name, ax) in enumerate(zip(class_names, axes.flat)):
    # One-versus-rest: True means "this class", False means "all other classes".
    actual_is_class = [label == class_index for label in y_true]
    predicted_is_class = [label == class_index for label in y_pred]
    binary_cm = confusion_matrix(
        actual_is_class, predicted_is_class, labels=[False, True]
    )
    tn, fp, fn, tp = binary_cm.ravel()
    assert tn + fp + fn + tp == len(y_true)
    print(f"{class_name:10} | TP={tp:3d} FP={fp:3d} FN={fn:3d} TN={tn:3d}")

    ConfusionMatrixDisplay(
        binary_cm, display_labels=["Other", class_name]
    ).plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(class_name)

plt.tight_layout()
plt.show()


## بررسی خطاهای با اطمینان بالا

در ماتریس خطا، سطر برچسب واقعی و ستون پیش‌بینی مدل است. ۱۳ کامیون به کامیونت و ۹ کامیونت به کامیون اشتباه شده‌اند. ۸ کامیونت نیز آمبولانس پیش‌بینی شده‌اند. این اعداد به‌تنهایی علت اشتباه را ثابت نمی‌کنند؛ باید تصاویر را بررسی کنیم.

سلول بعدی تمام موارد اشتباه را پیدا می‌کند، آن‌ها را بر اساس احتمالِ کلاس پیش‌بینی‌شده از زیاد به کم مرتب می‌کند و ۱۲ مورد اول را نشان می‌دهد. این مقدار، اطمینان خروجی softmax است، نه احتمال تضمین‌شدهٔ درست‌بودن پاسخ. اگر برچسبی مشکوک بود، فایل خام را تغییر نده؛ ابتدا آن را در فهرست بازبینی محلی ثبت کنیم.


In [ ]:
# The validation DataLoader uses shuffle=False, so these positions match y_true/y_pred.
wrong_indices = [
    i for i, (true_label, predicted_label) in enumerate(zip(y_true, y_pred))
    if true_label != predicted_label
]
wrong_indices.sort(
    key=lambda i: val_probabilities[i, y_pred[i]].item(),
    reverse=True,
)

print("Validation mistakes:", len(wrong_indices), "out of", len(y_true))
print("Highest-confidence mistakes shown:", min(12, len(wrong_indices)))

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, position in zip(axes.flat, wrong_indices[:12]):
    # Subset index -> full dataset index -> original image path.
    full_index = validation_subset.indices[position]
    image_path, _ = validation_full.samples[full_index]
    confidence = val_probabilities[position, y_pred[position]].item()

    with Image.open(image_path) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(
        f"True: {class_names[y_true[position]]} | "
        f"Pred: {class_names[y_pred[position]]}\n"
        f"Model confidence: {confidence:.1%}", fontsize=9
    )
    ax.axis("off")
for ax in list(axes.flat)[min(12, len(wrong_indices)):]:
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
position = wrong_indices[0]
full_index = validation_subset.indices[position]
image_path, _ = validation_full.samples[full_index]

print("Image:", image_path)
print("True:", class_names[y_true[position]])
print("Predicted:", class_names[y_pred[position]])

with Image.open(image_path) as image:
    display(image.copy())

## کدام دو کلاس بیشتر با هم اشتباه می‌شوند؟

طبق صورت پروژه، برای هر جفت کلاس، خطای دو جهت را از ماتریسِ نرمال‌شده با سطر واقعی جمع می‌کنیم:

```text
pair_confusion(i, j) = C_normalized[i, j] + C_normalized[j, i]
```

عدد اول می‌گوید چه سهمی از تصاویر واقعی کلاس `i` به اشتباه `j` پیش‌بینی شده‌اند؛ عدد دوم جهت عکس را نشان می‌دهد. این *امتیاز رتبه‌بندیِ جفت‌ها* است، نه درصد دقت مدل یا احتمال یک تصویر. بالا بودنش دلیل کافی برای ادغام کلاس‌ها یا جابه‌جایی فایل‌ها نیست؛ باید تصویر، تعریف کلاس و هزینهٔ خطا را هم بررسی کنیم.


In [ ]:
pair_results = []
for first in range(len(class_names)):
    for second in range(first + 1, len(class_names)):
        first_to_second = cm_normalized[first, second]
        second_to_first = cm_normalized[second, first]
        pair_results.append((
            first_to_second + second_to_first,
            first, second,
            first_to_second, second_to_first,
        ))

pair_results.sort(reverse=True)
for rank, (score, first, second, first_to_second, second_to_first) in enumerate(pair_results[:5], start=1):
    print(
        f"{rank}. {class_names[first]} ↔ {class_names[second]} | "
        f"{class_names[first]}→{class_names[second]}: "
        f"{cm_counts[first, second]}/{cm_counts[first].sum()} = {first_to_second:.3f} | "
        f"{class_names[second]}→{class_names[first]}: "
        f"{cm_counts[second, first]}/{cm_counts[second].sum()} = {second_to_first:.3f} | "
        f"pair score: {score:.3f}"
    )


## ResNet18، قدم ۱: معماری مدل ازپیش‌آموزش‌دیده

طبق صورت پروژه، باید یک ResNet18 با وزن‌های آموزش‌دیده روی ImageNet را برای **استخراج ویژگی** و سپس **fine-tuning لایهٔ `layer4`** مقایسه کنیم. فعلاً فقط مدل را می‌سازیم و لایهٔ ورودی و لایهٔ آخرش را می‌بینیم؛ این سلول **هیچ آموزشی انجام نمی‌دهد** و وزن‌ها را تغییر نمی‌دهد. فایل وزن ImageNet در کش محلی موجود است.

سؤال کلیدی: لایهٔ `fc` فعلی ۵۱۲ ویژگی ورودی می‌گیرد و ۱۰۰۰ score می‌دهد. این ۱۰۰۰ خروجی برای مسئلهٔ ما نیستند؛ در قدم بعد سرِ مدل را با یک لایهٔ ۸خروجی جایگزین می‌کنیم. طبق صورت پروژه، `conv1` و `maxpool` اصلی را برای مدل pretrained دست‌نخورده می‌گذاریم و از preprocessing سازگار با ImageNet استفاده می‌کنیم.


In [ ]:
import json
from pathlib import Path
from torchvision import models

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
resnet_manifest = json.loads((project_root / "reports/base_split.json").read_text())

resnet_weights = models.ResNet18_Weights.DEFAULT
resnet18_model = models.resnet18(weights=resnet_weights)

print("First convolution:", resnet18_model.conv1)
print("Last layer:", resnet18_model.fc)
print("Features entering fc:", resnet18_model.fc.in_features)
print("Original output scores:", resnet18_model.fc.out_features)
print("Our vehicle classes:", len(resnet_manifest["class_to_idx"]))
print("ImageNet preprocessing:", resnet_weights.transforms())


## ResNet18، قدم ۲: backbone ثابت، سرِ جدید قابل‌آموزش

ترتیب این سه کار مهم است و از فایل تدریس استاد پیروی می‌کند: **(۱) بارگذاری وزن pretrained، (۲) فریز کردن پارامترهای موجود، (۳) جایگزین کردن `fc` با یک لایهٔ تازهٔ ۸خروجی**. اگر لایهٔ تازه را قبل از فریز کردن بسازیم، آن هم فریز می‌شود. لایهٔ تازه به‌صورت پیش‌فرض `requires_grad=True` دارد.

در این مرحله فقط مدل را آماده می‌کنیم؛ هنوز هیچ batch، loss، `backward()` یا `optimizer.step()` نداریم. backbone در forward همچنان ویژگی تولید می‌کند، اما وزن‌هایش در آموزشِ مرحلهٔ اول تغییر نمی‌کنند. انتظار داریم تعداد پارامترهای قابل‌آموزش فقط `512 × 8 + 8 = 4104` باشد: برای هر یک از ۸ کلاس، ۵۱۲ وزن و یک bias. `eval()` کار فریز کردن را انجام نمی‌دهد؛ رفتار BatchNorm را در قدم‌های بعد جداگانه تنظیم می‌کنیم.


In [ ]:
from torch import nn

# Same pretrained architecture as the inspection cell above.
feature_model = models.resnet18(weights=resnet_weights)

# Freeze every parameter that came with the pretrained model.
feature_model.requires_grad_(False)

num_features = feature_model.fc.in_features
num_classes = len(resnet_manifest["class_to_idx"])
feature_model.fc = nn.Linear(num_features, num_classes)

trainable_names = [
    name for name, parameter in feature_model.named_parameters()
    if parameter.requires_grad
]
total_parameters = sum(parameter.numel() for parameter in feature_model.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in feature_model.parameters()
    if parameter.requires_grad
)

assert trainable_names == ["fc.weight", "fc.bias"]
assert trainable_parameters == num_features * num_classes + num_classes
print("New classifier:", feature_model.fc)
print("Backbone conv1 trainable:", feature_model.conv1.weight.requires_grad)
print("Classifier weight trainable:", feature_model.fc.weight.requires_grad)
print("Trainable parameter names:", trainable_names)
print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)


## ResNet18، قدم ۳: تصویر را برای وزن‌های ImageNet آماده کنیم

در CNN خودمان تصویر را با padding به `128×128` رساندیم و با میانگین/انحراف معیار `0.5` نرمال کردیم. وزن‌های آمادهٔ ResNet18 با روش دیگری آموزش دیده‌اند؛ بنابراین آن pipeline را بدون بررسی برای ResNet تکرار نمی‌کنیم. دستور `resnet_weights.transforms()` نسخهٔ ارزیابیِ سازگار با همین وزن‌ها را می‌دهد: ضلع کوتاه تصویر به ۲۵۶ می‌رسد، مرکز آن به `224×224` بریده می‌شود، پیکسل‌ها tensor می‌شوند و هر کانال با آمار ImageNet نرمال می‌شود.

سلول زیر یک **عکس از بخش آموزش** را قبل و بعد از `Resize → CenterCrop` نشان می‌دهد و شکل tensor نهایی را چاپ می‌کند. عکس نمایش‌داده‌شده دادهٔ خصوصی است و خروجی تصویری نوت‌بوک وارد Git نمی‌شود. هنوز مدل را آموزش یا ارزیابی نمی‌کنیم. به این دقت کن که بریدن مرکز تصویر ممکن است در عکس‌های باریک بخشی از خودرو را حذف کند؛ بعد از دیدن نمونه دربارهٔ روش مناسب دادهٔ خودمان تصمیم می‌گیریم.


In [ ]:
from PIL import Image
from IPython.display import display
from torchvision import transforms
from torchvision.transforms import InterpolationMode

resnet_eval_transform = resnet_weights.transforms()

# Choose a tall training image so any center-crop loss is easy to see.
sample_image_path = None
for relative_path in resnet_manifest["training_paths"]:
    candidate = project_root / relative_path
    with Image.open(candidate) as image:
        width, height = image.size
    if max(width, height) / min(width, height) >= 1.8:
        sample_image_path = candidate
        break
if sample_image_path is None:
    sample_image_path = project_root / resnet_manifest["training_paths"][0]

with Image.open(sample_image_path) as image:
    original_image = image.convert("RGB")

# For display only: the geometric part before ToTensor and Normalize.
geometry_only = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BILINEAR),
    transforms.CenterCrop(224),
])
preview_image = geometry_only(original_image)
resnet_input = resnet_eval_transform(original_image)

print("Original size (width, height):", original_image.size)
print("After resize and center crop:", preview_image.size)
print("Tensor shape (channels, height, width):", tuple(resnet_input.shape))
print("Tensor value range after ImageNet normalization:",
      round(resnet_input.min().item(), 2), round(resnet_input.max().item(), 2))
print("Original image, then model-visible crop:")
display(original_image, preview_image)


## ResNet18، قدم ۴: transform سادهٔ درس استاد

برای مسیر اصلی، آزمایش جانبیِ padding را فعلاً کنار می‌گذاریم. در کد آموزشی استاد، تصویر به‌صورت مستقیم به `224×224` تغییر اندازه داده می‌شود، به tensor تبدیل می‌شود و با میانگین و انحراف معیار ImageNet نرمال می‌شود. این روش تمام تصویر را نگه می‌دارد و center crop ندارد؛ در عکس‌های کشیده ممکن است نسبت طول‌به‌عرض را تغییر دهد. فعلاً همین روش ساده را برای آموزش و validation استفاده می‌کنیم تا feature extraction و سپس fine-tuning را یاد بگیریم. augmentation و مقایسهٔ روش‌های دیگر را بعداً به‌صورت آزمایش جدا انجام می‌دهیم.

سلول بعد فقط transform را تعریف می‌کند و روی همان یک عکس آموزشی شکل tensor را بررسی می‌کند. وزن‌های مدل تغییر نمی‌کنند و هنوز DataLoader یا حلقهٔ آموزش نداریم.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

transformed_sample = resnet_transform(original_image)
print("Original size (width, height):", original_image.size)
print("ResNet tensor shape (channels, height, width):", tuple(transformed_sample.shape))
print("Same normalization as pretrained weights:",
      IMAGENET_MEAN == list(resnet_eval_transform.mean)
      and IMAGENET_STD == list(resnet_eval_transform.std))


## ResNet18، قدم ۵: همان تقسیم واحد را بخوانیم

CNN و ResNet هر دو فهرست آموزش و validation را از یک manifest خصوصی می‌خوانند. این فهرست از ابتدا شامل دو دیتاست، نمونه‌های مجاز `unclean` و نیسان با برچسب هدف `vanet` است. برای سنجش جداگانهٔ نیسان، فقط ردیف‌های نیسانِ **داخل validation** را انتخاب می‌کنیم؛ آن‌ها دوباره به مجموعهٔ آموزش اضافه نمی‌شوند. `fingerprint` کمک می‌کند نتیجه و checkpoint را به تقسیم درست نسبت دهیم.


In [ ]:
import sys
from collections import Counter

sys.path.insert(0, str(project_root))
from scripts.run_experiments import manifest_data

resnet_manifest, resnet_training_rows, resnet_validation_rows = manifest_data()
resnet_neysan_validation_rows = [
    row for row in resnet_validation_rows if row["original_label"] == "neysan"
]
assert resnet_training_rows and resnet_validation_rows and resnet_neysan_validation_rows
print("Split fingerprint:", resnet_manifest["fingerprint"][:12])
print("Training:", len(resnet_training_rows))
print("Validation:", len(resnet_validation_rows))
print("Validation sources:", dict(Counter(row["source"] for row in resnet_validation_rows)))
print("Nissan in validation:", len(resnet_neysan_validation_rows))


## ResNet18، قدم ۶: Dataset و DataLoader

`VehicleDataset` هر مسیر و برچسب را هنگام درخواست به تصویر و اندیس کلاس تبدیل می‌کند. `DataLoader` تصاویر را در batchهای ۳۲تایی می‌خواند. برای آموزش `shuffle=True` و برای validation `shuffle=False` است. `resnet_neysan_loader` تنها نمایی از نیسان‌های داخل validation است؛ تعداد آن را با تعداد batchها اشتباه نگیر. این سلول فقط یک batch آزمایشی می‌خواند و وزن‌ها را تغییر نمی‌دهد.


In [ ]:
import torch
from torch.utils.data import DataLoader
from scripts.run_experiments import VehicleDataset

resnet_train_dataset = VehicleDataset(
    resnet_training_rows, resnet_transform, resnet_manifest["class_to_idx"]
)
resnet_val_dataset = VehicleDataset(
    resnet_validation_rows, resnet_transform, resnet_manifest["class_to_idx"]
)
resnet_neysan_dataset = VehicleDataset(
    resnet_neysan_validation_rows, resnet_transform, resnet_manifest["class_to_idx"]
)
resnet_train_loader = DataLoader(
    resnet_train_dataset, batch_size=32, shuffle=True,
    generator=torch.Generator().manual_seed(42), num_workers=0,
)
resnet_val_loader = DataLoader(
    resnet_val_dataset, batch_size=32, shuffle=False, num_workers=0,
)
resnet_neysan_loader = DataLoader(
    resnet_neysan_dataset, batch_size=32, shuffle=False, num_workers=0,
)
batch_images, batch_labels = next(iter(resnet_train_loader))
print("Dataset lengths:", len(resnet_train_dataset), len(resnet_val_dataset), len(resnet_neysan_dataset))
print("Batches per epoch:", len(resnet_train_loader))
print("Images shape [batch, RGB, height, width]:", tuple(batch_images.shape))
print("Labels shape [batch]:", tuple(batch_labels.shape))
print("First five label indices:", batch_labels[:5].tolist())


## ResNet18، قدم ۷: یک forward pass و محاسبهٔ loss

ابتدا مدل و یک batch را روی یک device می‌گذاریم. برای آنکه آمار داخلی BatchNormِ backbone ثابت بماند، کل مدل را در حالت `eval()` می‌گذاریم و سپس head را با `fc.train()` در حالت آموزش می‌گذاریم؛ این حالت‌ها **فریزِ پارامتر نیستند**. پارامترهای backbone قبلاً با `requires_grad_(False)` فریز شدند و سرِ تازه قابل‌آموزش است. از `torch.no_grad()` استفاده نمی‌کنیم، چون در قدم بعد می‌خواهیم گرادیانِ سر را با `backward()` حساب کنیم.

خروجی `feature_model(images)` باید برای ۳۲ عکس، شکل `[32, 8]` داشته باشد: در هر ردیف هشت **logit/score خام**، نه هشت احتمال. `CrossEntropyLoss` این scoreها و شمارهٔ کلاس درست را می‌گیرد؛ نباید پیش از آن خودمان `softmax` بزنیم. این سلول فقط forward و loss را حساب می‌کند، هیچ وزنی را تغییر نمی‌دهد.


In [ ]:
from scripts.run_experiments import device_for_run

resnet_device = device_for_run()
feature_model = feature_model.to(resnet_device)
feature_model.eval()       # Keep frozen backbone BatchNorm statistics fixed.
feature_model.fc.train()   # The new classifier is the training part.

images_for_resnet = batch_images.to(resnet_device)
labels_for_resnet = batch_labels.to(resnet_device)
class_scores = feature_model(images_for_resnet)
loss_fn = nn.CrossEntropyLoss()
batch_loss = loss_fn(class_scores, labels_for_resnet)

assert class_scores.shape == (len(batch_labels), len(resnet_manifest["class_to_idx"]))
print("Device:", resnet_device)
print("Scores shape [batch, classes]:", tuple(class_scores.shape))
print("One image's eight raw scores:",
      [round(value, 3) for value in class_scores[0].detach().cpu().tolist()])
print("That image's true label index:", labels_for_resnet[0].item())
print("Batch loss:", round(batch_loss.item(), 4))
print("Scores require gradient:", class_scores.requires_grad)
print("Backbone training mode:", feature_model.layer4.training)
print("Classifier training mode:", feature_model.fc.training)


## ResNet18، قدم ۸: گرادیانِ سرِ تازه را بررسی کنیم

`backward()` مشتق loss را برای پارامترهای قابل‌آموزش حساب می‌کند؛ **به‌تنهایی وزن‌ها را تغییر نمی‌دهد**. ابتدا `zero_grad(set_to_none=True)` گرادیان باقی‌مانده از اجرای قبلی سلول را پاک می‌کند. سپس forward و loss را **دوباره** می‌سازیم تا اگر این سلول را تکرار کردی، graph تازه‌ای برای `backward()` داشته باشیم. در پایان باید `.grad` در `conv1` و `layer4` برابر `None` باشد، ولی در `fc.weight` و `fc.bias` وجود داشته باشد. شکل گرادیان `fc.weight` باید `[8, 512]` باشد: به ازای هر کلاس، مشتقِ ۵۱۲ وزن.

این سلول هنوز optimizer و `optimizer.step()` ندارد؛ پس حتی با وجود گرادیانِ `fc`، وزن‌ها همان مقادیر قبلی می‌مانند. این دقیقاً تفاوتِ «محاسبهٔ جهت تغییر» و «اعمال تغییر» است.


In [ ]:
feature_model.eval()
feature_model.fc.train()
feature_model.zero_grad(set_to_none=True)

head_weights_before = feature_model.fc.weight.detach().clone()
fresh_scores = feature_model(images_for_resnet)
fresh_loss = loss_fn(fresh_scores, labels_for_resnet)
fresh_loss.backward()

assert feature_model.conv1.weight.grad is None
assert feature_model.layer4[0].conv1.weight.grad is None
assert feature_model.fc.weight.grad is not None
assert feature_model.fc.bias.grad is not None

print("Conv1 requires_grad:", feature_model.conv1.weight.requires_grad)
print("Conv1 gradient:", feature_model.conv1.weight.grad)
print("Layer4 gradient:", feature_model.layer4[0].conv1.weight.grad)
print("FC requires_grad:", feature_model.fc.weight.requires_grad)
print("FC weight gradient shape:", tuple(feature_model.fc.weight.grad.shape))
print("FC bias gradient shape:", tuple(feature_model.fc.bias.grad.shape))
print("FC gradient norm:", round(feature_model.fc.weight.grad.norm().item(), 4))
print("FC weights unchanged after backward:",
      torch.equal(head_weights_before, feature_model.fc.weight.detach()))


## ResNet18، قدم ۹: یک به‌روزرسانی واقعی برای `fc`

در قدم قبل `backward()` فقط گرادیان را محاسبه کرد و وزن‌ها دست‌نخورده ماندند. این بار یک optimizer از نوع Adam می‌سازیم که **فقط** پارامترهای `feature_model.fc` را دریافت می‌کند. بعد از پاک‌کردن گرادیان‌های قبلی، برای همان batch یک forward و backward تازه انجام می‌دهیم و با `optimizer.step()` وزن‌های head را یک‌بار تغییر می‌دهیم. برای مقایسه، loss همان batch را پس از تغییر دوباره محاسبه می‌کنیم؛ این یک آزمایش آموزشی است و معیار عملکرد روی validation نیست.

این سلول را یک‌بار اجرا کن: هر اجرای دوباره، یک گام آموزشی دیگر روی همان batch خواهد بود.


In [ ]:
head_optimizer = torch.optim.Adam(feature_model.fc.parameters(), lr=1e-3)
feature_model.eval()
feature_model.fc.train()
head_optimizer.zero_grad(set_to_none=True)

head_weights_before_step = feature_model.fc.weight.detach().clone()
scores_before_step = feature_model(images_for_resnet)
loss_before_step = loss_fn(scores_before_step, labels_for_resnet)
loss_before_step.backward()
head_optimizer.step()

with torch.no_grad():
    scores_after_step = feature_model(images_for_resnet)
    loss_after_step = loss_fn(scores_after_step, labels_for_resnet)

weight_change = (feature_model.fc.weight.detach() - head_weights_before_step).norm().item()
assert weight_change > 0
assert feature_model.conv1.weight.grad is None

print("Loss before step:", round(loss_before_step.item(), 4))
print("Loss after step on the same batch:", round(loss_after_step.item(), 4))
print("FC weight change (L2 norm):", round(weight_change, 6))
print("Backbone conv1 still has no gradient:", feature_model.conv1.weight.grad is None)


## ResNet18، قدم ۱۰: یک epoch فقط برای سرِ طبقه‌بندی

تابع قبلیِ `train_one_epoch` برای CNN از `model.train()` استفاده می‌کرد. این بار می‌خواهیم backbone پیش‌آموزش‌دیده و آمار BatchNorm آن ثابت بمانند؛ برای همین ابتدای تابع `model.eval()` و سپس `model.fc.train()` را صدا می‌زنیم. فریز وزن‌ها همچنان به `requires_grad_(False)` مربوط است، نه به `eval()`.

بقیهٔ حلقه همان الگویی است که در CNN خواندیم: هر batch را به device می‌بریم، گرادیان قبلی را پاک می‌کنیم، forward و loss را حساب می‌کنیم، `backward()` و `optimizer.step()` را اجرا می‌کنیم، سپس loss و تعداد پیش‌بینی‌های درست را برای کل epoch جمع می‌کنیم. این سلول فقط تابع را تعریف می‌کند؛ با اجرای آن هنوز epoch آغاز نمی‌شود.


In [ ]:
def train_feature_head_one_epoch(model, loader, optimizer, loss_fn, device):
    model.eval()       # Keep pretrained backbone BatchNorm statistics fixed.
    model.fc.train()   # Train the new classification head.

    total_loss = 0.0
    correct_predictions = 0
    total_images = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        scores = model(images)
        loss = loss_fn(scores, labels)
        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        correct_predictions += (scores.argmax(dim=1) == labels).sum().item()
        total_images += batch_size

    return {
        "loss": total_loss / total_images,
        "accuracy": correct_predictions / total_images,
    }


## ResNet18، قدم ۱۱: مدل تازه و اولین epoch

مدلِ قدم‌های ۷ تا ۹ برای آزمایش یک batch بود و `fc` آن یک‌بار تغییر کرد. برای آزمایش آموزشی یک مدل تازه با وزن‌های پیش‌آموزش‌دیده می‌سازیم، backbone را فریز می‌کنیم، و یک سرِ ۸کلاسهٔ تازه می‌گذاریم. optimizer فقط پارامترهای همین سر را می‌گیرد. سپس **یک epoch کامل** روی تصویرهای مجموعهٔ آموزش فعلی اجرا می‌کنیم. ارزیابی روی تصویرهای validation فعلی در سلول بعدی و جدا از آموزش انجام می‌شود.

این سلول زمان‌برتر از سلول‌های قبلی است: batchهای آموزشی را یک‌بار پردازش می‌کند. اگر دوباره اجرا شود، مدل و optimizer از نو ساخته می‌شوند و همان آزمایش یک‌epochی تکرار می‌شود.


In [ ]:
torch.manual_seed(42)
feature_train_model = models.resnet18(weights=resnet_weights)
feature_train_model.requires_grad_(False)
feature_train_model.fc = nn.Linear(
    feature_train_model.fc.in_features,
    len(resnet_manifest["class_to_idx"]),
)
feature_train_model = feature_train_model.to(resnet_device)
feature_optimizer = torch.optim.Adam(feature_train_model.fc.parameters(), lr=1e-3)

feature_train_metrics = train_feature_head_one_epoch(
    feature_train_model, resnet_train_loader, feature_optimizer, loss_fn, resnet_device
)
print("Train loss:", round(feature_train_metrics["loss"], 4))
print("Train accuracy:", f'{feature_train_metrics["accuracy"]:.1%}')


## ResNet18، قدم ۱۲: ارزیابی جداگانه روی validation

در این سلول مدلِ آموزش‌دیدهٔ قدم قبل را با `eval()` در حالت ارزیابی قرار می‌دهیم. `torch.no_grad()` ساخت گراف گرادیان را برای validation غیرفعال می‌کند؛ هیچ `backward()` یا `optimizer.step()` نداریم، پس وزن‌ها تغییر نمی‌کنند. برای هر batch، loss را در تعداد تصاویر آن ضرب می‌کنیم و در پایان بر کل تصاویر تقسیم می‌کنیم؛ accuracy هم تعداد پیش‌بینی‌های درست تقسیم بر کل تصاویر است.

ارزیابی را در سلول جدا قرار داده‌ایم تا بتوانیم نتیجهٔ مدلِ فعلی را بدون اجرای دوبارهٔ epoch آموزشی ببینیم.


In [ ]:
feature_train_model.eval()
val_loss_sum = 0.0
val_correct = 0
val_total = 0

with torch.no_grad():
    for images, labels in resnet_val_loader:
        images = images.to(resnet_device)
        labels = labels.to(resnet_device)
        scores = feature_train_model(images)
        loss = loss_fn(scores, labels)

        batch_size = images.size(0)
        val_loss_sum += loss.item() * batch_size
        val_correct += (scores.argmax(dim=1) == labels).sum().item()
        val_total += batch_size

feature_val_metrics = {
    "loss": val_loss_sum / val_total,
    "accuracy": val_correct / val_total,
}
print("Train loss:", round(feature_train_metrics["loss"], 4))
print("Train accuracy:", f'{feature_train_metrics["accuracy"]:.1%}')
print("Validation loss:", round(feature_val_metrics["loss"], 4))
print("Validation accuracy:", f'{feature_val_metrics["accuracy"]:.1%}')


## ResNet18، قدم ۱۳: تابع ارزیابیِ قابل‌استفاده در هر epoch

محاسبهٔ validation در قدم قبل را به یک تابع تبدیل می‌کنیم تا بعد از هر epoch دقیقاً همان روش را به کار ببریم. `model.eval()` رفتار BatchNorm را ثابت می‌گذارد و `torch.no_grad()` ساخت گراف گرادیان را غیرفعال می‌کند. این تابع optimizer ندارد و فقط loss و accuracy را برمی‌گرداند؛ با اجرای این سلول هنوز آموزش تازه‌ای آغاز نمی‌شود.


In [ ]:
def evaluate_feature_head(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    correct_predictions = 0
    total_images = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            scores = model(images)
            loss = loss_fn(scores, labels)

            batch_size = images.size(0)
            total_loss += loss.item() * batch_size
            correct_predictions += (scores.argmax(dim=1) == labels).sum().item()
            total_images += batch_size

    return {
        "loss": total_loss / total_images,
        "accuracy": correct_predictions / total_images,
    }


## ResNet18، قدم ۱۴: ادامهٔ آموزش سرِ طبقه‌بندی تا تعداد epoch انتخاب‌شده

نتیجهٔ epoch اول از `feature_train_metrics` و `feature_val_metrics` وارد `feature_history` می‌شود. پس حلقه از **epoch دوم** شروع می‌کند و مدل و optimizer موجود را ادامه می‌دهد؛ هیچ‌کدام در این سلول از نو ساخته نمی‌شوند. پس از هر epoch، train و validation را جداگانه محاسبه و ثبت می‌کنیم.

طبق الگوی فایل استاد، هر وقت **validation loss کمتر** شد، با `deepcopy(model.state_dict())` یک کپی مستقل از همهٔ وزن‌ها و بافرهای مدل نگه می‌داریم. در پایان، `load_state_dict` بهترین وضعیت را برمی‌گرداند. ممکن است epoch دارای کمترین validation loss با epoch دارای بیشترین validation accuracy یکسان نباشد؛ معیار انتخاب ما در این مرحله loss است. در اجرای ثبت‌شده، `feature_epochs=30` است و این سلول بعد از epoch اول، ۲۹ epoch دیگر اجرا می‌کند؛ بنابراین زمان‌بر است. برای تکرار آزمایش از ابتدا، سلول ساخت مدل و epoch اول را هم دوباره اجرا کن.


In [ ]:
from copy import deepcopy

feature_epochs = 30
feature_history = [{
    "epoch": 1,
    "train_loss": feature_train_metrics["loss"],
    "train_accuracy": feature_train_metrics["accuracy"],
    "val_loss": feature_val_metrics["loss"],
    "val_accuracy": feature_val_metrics["accuracy"],
}]
best_feature_val_loss = feature_val_metrics["loss"]
best_feature_epoch = 1
best_feature_state = deepcopy(feature_train_model.state_dict())

print(
    f"Epoch 01/{feature_epochs} | "
    f"train loss: {feature_train_metrics['loss']:.3f} | "
    f"val loss: {feature_val_metrics['loss']:.3f} | "
    f"train acc: {feature_train_metrics['accuracy']:.1%} | "
    f"val acc: {feature_val_metrics['accuracy']:.1%}"
)

for epoch in range(2, feature_epochs + 1):
    train_metrics = train_feature_head_one_epoch(
        feature_train_model, resnet_train_loader, feature_optimizer, loss_fn, resnet_device
    )
    val_metrics = evaluate_feature_head(
        feature_train_model, resnet_val_loader, loss_fn, resnet_device
    )
    feature_history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
    })

    if val_metrics["loss"] < best_feature_val_loss:
        best_feature_val_loss = val_metrics["loss"]
        best_feature_epoch = epoch
        best_feature_state = deepcopy(feature_train_model.state_dict())

    print(
        f"Epoch {epoch:02d}/{feature_epochs} | "
        f"train loss: {train_metrics['loss']:.3f} | "
        f"val loss: {val_metrics['loss']:.3f} | "
        f"train acc: {train_metrics['accuracy']:.1%} | "
        f"val acc: {val_metrics['accuracy']:.1%}"
    )

feature_train_model.load_state_dict(best_feature_state)
print("Best epoch by validation loss:", best_feature_epoch)
print("Best validation loss:", round(best_feature_val_loss, 4))


## ثبت و مقایسهٔ نتایج

نتایج قبلی در `reports/experiment_results.json` با وضعیت `historical_split` مشخص شده‌اند. فقط نتیجه‌هایی را با هم مقایسه کن که `split_fingerprint` یکسان دارند. بعد از اجرای تازهٔ CNN و ResNet روی تقسیم واحد، معیارهای جدید را در همین فایل ثبت می‌کنیم؛ اعداد قبلی را به تقسیم جدید نسبت نمی‌دهیم. دقت validation داخلی، نتیجهٔ تست مخفی منتورها نیست.


## ResNet18، قدم ۱۵: ذخیرهٔ بهترین وضعیت بر اساس validation loss

`best_feature_state` در حافظهٔ کرنل است. این سلول وزن‌های منتخب، پیش‌پردازش، ترتیب کلاس‌ها و شناسهٔ تقسیم داده را در یک checkpoint محلی ذخیره می‌کند. checkpoint داخل `checkpoints/` می‌ماند و به Git نمی‌رود. epoch و دقت منتخب را از خروجی **اجرای تازه** بخوان؛ عددهای اجرای قبلی به تقسیم دیگری تعلق داشتند.


In [ ]:
checkpoint_dir = project_root / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
feature_checkpoint_path = checkpoint_dir / "resnet18_feature_best_loss.pt"

checkpoint = {
    "architecture": "resnet18",
    "model_state_dict": {
        name: tensor.detach().cpu().clone()
        for name, tensor in feature_train_model.state_dict().items()
    },
    "class_to_idx": resnet_manifest["class_to_idx"],
    "split_fingerprint": resnet_manifest["fingerprint"],
    "preprocessing": {
        "resize": [224, 224],
        "mean": IMAGENET_MEAN,
        "std": IMAGENET_STD,
    },
    "training_mode": "fc_only",
    "selected_epoch": best_feature_epoch,
    "selection_metric": "validation_loss",
    "validation_loss": best_feature_val_loss,
    "validation_accuracy": feature_history[best_feature_epoch - 1]["val_accuracy"],
}
torch.save(checkpoint, feature_checkpoint_path)

print("Saved checkpoint:", feature_checkpoint_path)
print("Selected epoch:", checkpoint["selected_epoch"])
print("Validation loss:", round(checkpoint["validation_loss"], 4))
print("Validation accuracy:", f'{checkpoint["validation_accuracy"]:.1%}')

# Record only the metrics measured on this exact split.
feature_result = {
    "experiment": "resnet18_feature_extraction",
    "architecture": "resnet18", "training_mode": "fc_only",
    "split_fingerprint": resnet_manifest["fingerprint"],
    "result_status": "current_split",
    "training_images": len(resnet_training_rows),
    "validation_images": len(resnet_validation_rows),
    "epochs": len(feature_history),
    "selected_epoch": best_feature_epoch,
    "selection_metric": "validation_loss",
    "selected_validation_loss": best_feature_val_loss,
    "selected_validation_accuracy": feature_history[best_feature_epoch - 1]["val_accuracy"],
    "history": feature_history,
}
results_path = project_root / "reports/experiment_results.json"
all_results = json.loads(results_path.read_text()) if results_path.exists() else {}
all_results["_split"] = {
    "current_fingerprint": resnet_manifest["fingerprint"],
    "comparison_rule": "Compare results only when split_fingerprint matches.",
}
all_results["resnet18_feature_extraction"] = feature_result
results_path.write_text(json.dumps(all_results, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Recorded current feature-extraction result in:", results_path.name)



## ResNet18، قدم ۱۶: آماده‌سازی fine-tuning برای `layer4` و `fc`

در مرحلهٔ قبل، فقط `fc` آموزش دید. اکنون از همان مدلِ منتخبِ feature extraction یک **کپی مستقل** می‌سازیم تا مدل قبلی برای مقایسه دست‌نخورده بماند. اگر هنوز سلول checkpoint قدم ۱۵ را اجرا نکرده‌ای، پیش از ادامه آن را اجرا کن تا وزن‌های فعلی روی دیسک هم محفوظ باشند.

روی کپی، `layer4` و `fc` را قابل‌آموزش می‌کنیم. لایه‌های `conv1` تا `layer3` همچنان فریز می‌مانند. طبق الگوی فایل استاد، پارامترهای تمام `BatchNorm2d`ها را دوباره فریز می‌کنیم؛ چون `layer4.requires_grad_(True)` در ابتدا BatchNormهای داخل خودِ `layer4` را هم فعال می‌کند. `eval()` روی BatchNorm، رفتار آمار داخلی آن را تنظیم می‌کند و جایگزینِ فریز کردن پارامترهایش نیست. در سلول بعدی باید این حالت را هنگام هر epoch نیز حفظ کنیم.

این سلول فقط انتخاب و شمارش پارامترهاست؛ optimizer و آموزش هنوز شروع نمی‌شوند.


In [ ]:
from copy import deepcopy

fine_tune_model = deepcopy(feature_train_model)
fine_tune_model.layer4.requires_grad_(True)
fine_tune_model.fc.requires_grad_(True)

for module in fine_tune_model.modules():
    if isinstance(module, nn.BatchNorm2d):
        module.requires_grad_(False)
        module.eval()

fine_tune_groups = {
    "conv1": fine_tune_model.conv1,
    "layer1": fine_tune_model.layer1,
    "layer2": fine_tune_model.layer2,
    "layer3": fine_tune_model.layer3,
    "layer4": fine_tune_model.layer4,
    "fc": fine_tune_model.fc,
}
for group_name, module in fine_tune_groups.items():
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    total = sum(p.numel() for p in module.parameters())
    print(f"{group_name:7s} | trainable: {trainable:>10,} / total: {total:>10,}")

trainable_names = [
    name for name, parameter in fine_tune_model.named_parameters()
    if parameter.requires_grad
]
assert trainable_names
assert all(name.startswith(("layer4.", "fc.")) for name in trainable_names)
assert all(
    not parameter.requires_grad
    for module in fine_tune_model.modules()
    if isinstance(module, nn.BatchNorm2d)
    for parameter in module.parameters(recurse=False)
)
print("Total trainable parameters:", sum(p.numel() for p in fine_tune_model.parameters() if p.requires_grad))
print("BatchNorm parameters trainable:", 0)


## ResNet18، قدم ۱۷: optimizer با دو نرخ یادگیری

در قدم قبل فقط `requires_grad` پارامترها را تنظیم کردیم. optimizer قدیمیِ `feature_optimizer` فقط `fc` را می‌شناخت؛ باز کردن `layer4` آن را خودکار به optimizer اضافه نمی‌کند. پس یک optimizer تازه با دو گروه می‌سازیم: کانولوشن‌های پیش‌آموزش‌دیدهٔ `layer4` با نرخ یادگیری کوچک‌تر `1e-4` و سرِ طبقه‌بندی `fc` با نرخ `1e-3`. BatchNormها به‌خاطر شرط `requires_grad` وارد گروه `layer4` نمی‌شوند. `weight_decay=1e-4` هم مطابق نمونهٔ استاد به optimizer داده شده است.

پایان سلول بررسی می‌کند هیچ پارامتری در هر دو گروه نباشد، هیچ پارامتر قابل‌آموزشی جا نمانده باشد، و هیچ پارامتری دوبار وارد optimizer نشده باشد. ساخت optimizer **به‌تنهایی وزن‌ها را تغییر نمی‌دهد**؛ هنوز `backward()` و `step()` اجرا نکرده‌ایم.


In [ ]:
LAYER4_LR = 1e-4
HEAD_LR = 1e-3

layer4_parameters = [
    parameter
    for parameter in fine_tune_model.layer4.parameters()
    if parameter.requires_grad
]
head_parameters = [
    parameter
    for parameter in fine_tune_model.fc.parameters()
    if parameter.requires_grad
]

fine_tune_optimizer = torch.optim.Adam(
    [
        {"params": layer4_parameters, "lr": LAYER4_LR, "name": "layer4"},
        {"params": head_parameters, "lr": HEAD_LR, "name": "fc"},
    ],
    weight_decay=1e-4,
)

optimizer_parameters = [
    parameter
    for group in fine_tune_optimizer.param_groups
    for parameter in group["params"]
]
trainable_parameter_ids = {
    id(parameter)
    for parameter in fine_tune_model.parameters()
    if parameter.requires_grad
}
assert set(map(id, layer4_parameters)).isdisjoint(map(id, head_parameters))
assert len(optimizer_parameters) == len({id(parameter) for parameter in optimizer_parameters})
assert {id(parameter) for parameter in optimizer_parameters} == trainable_parameter_ids

for group in fine_tune_optimizer.param_groups:
    count = sum(parameter.numel() for parameter in group["params"])
    print(f"{group['name']:6s} | lr={group['lr']:.0e} | parameters={count:,}")
print("All trainable parameters are in the optimizer exactly once.")


## ResNet18، قدم ۱۸: یک batch آزمایشی پیش از fine-tuning کامل

برای آموزش، کل مدل را با `train()` در حالت آموزش می‌گذاریم؛ سپس همهٔ `BatchNorm2d`ها را دوباره به `eval()` برمی‌گردانیم تا آمار ذخیره‌شدهٔ آن‌ها تغییر نکند. `eval()` روی BatchNorm مسیر گرادیان را برای کانولوشن‌های `layer4` قطع نمی‌کند. با `zero_grad` گرادیان‌های قبلی را پاک می‌کنیم و از یک batch، خروجی، loss و `backward()` می‌گیریم.

باید ببینیم `conv1` گرادیان ندارد، اما convolution داخل `layer4` و وزن‌های `fc` گرادیان دارند. همچنین `running_mean` یک BatchNorm نمونه را قبل و بعد از forward مقایسه می‌کنیم. این سلول **`optimizer.step()` ندارد**؛ پس وزن‌ها و وضعیت Adam هنوز تغییر نمی‌کنند و آزمایش اصلی از همان وزن‌های feature extraction شروع می‌شود.


In [ ]:
fine_tune_model.train()
for module in fine_tune_model.modules():
    if isinstance(module, nn.BatchNorm2d):
        module.eval()

assert all(
    not module.training
    for module in fine_tune_model.modules()
    if isinstance(module, nn.BatchNorm2d)
)

sample_bn = fine_tune_model.layer4[0].bn1
running_mean_before = sample_bn.running_mean.detach().clone()
fine_tune_optimizer.zero_grad(set_to_none=True)

images, labels = next(iter(resnet_train_loader))
images = images.to(resnet_device)
labels = labels.to(resnet_device)
scores = fine_tune_model(images)
loss = loss_fn(scores, labels)
loss.backward()

assert fine_tune_model.conv1.weight.grad is None
assert fine_tune_model.layer4[0].conv1.weight.grad is not None
assert fine_tune_model.fc.weight.grad is not None
assert torch.equal(running_mean_before, sample_bn.running_mean)

print("One-batch loss:", round(loss.item(), 4))
print("Frozen conv1 gradient:", fine_tune_model.conv1.weight.grad)
print("Layer4 gradient norm:", round(fine_tune_model.layer4[0].conv1.weight.grad.norm().item(), 4))
print("FC gradient norm:", round(fine_tune_model.fc.weight.grad.norm().item(), 4))
print("BatchNorm running mean unchanged:", torch.equal(running_mean_before, sample_bn.running_mean))


## ResNet18، قدم ۱۹: تابع یک epoch برای fine-tuning

حلقهٔ آموزش شبیه مرحلهٔ `fc`-only است، اما اکنون `layer4` نیز گرادیان و به‌روزرسانی دارد. ابتدای هر epoch، `model.train()` را صدا می‌زنیم و بلافاصله تمام `BatchNorm2d`ها را دوباره `eval()` می‌کنیم؛ وگرنه آمار ذخیره‌شدهٔ آن‌ها در batchهای آموزش تغییر می‌کند. `requires_grad_(False)` پارامترهای BatchNorm از قدم ۱۶ همچنان برقرار است.

برای هر batch: انتقال به device، پاک‌کردن گرادیان قبلی، forward، loss، `backward()` و `optimizer.step()` انجام می‌شود. در پایان، میانگین loss و دقت کل epoch را برمی‌گردانیم. اجرای این سلول فقط تابع را تعریف می‌کند.


In [ ]:
def train_fine_tune_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

    total_loss = 0.0
    correct_predictions = 0
    total_images = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        scores = model(images)
        loss = loss_fn(scores, labels)
        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        correct_predictions += (scores.argmax(dim=1) == labels).sum().item()
        total_images += batch_size

    return {
        "loss": total_loss / total_images,
        "accuracy": correct_predictions / total_images,
    }


## ResNet18، قدم ۲۰: اولین epoch واقعی fine-tuning

اکنون یک epoch کامل روی تصویرهای مجموعهٔ آموزش فعلی اجرا می‌کنیم و با تابع `evaluate_feature_head` روی تصویرهای validation فعلی می‌سنجیم. پیش از شروع، وزنِ یک convolution از `layer4` و آمار یک BatchNorm را کپی می‌کنیم. پس از epoch باید وزن convolution تغییر کرده باشد، ولی `running_mean` آن BatchNorm ثابت مانده باشد. مدلِ فقط-`fc` و checkpoint آن دست‌نخورده می‌مانند.

این سلول همهٔ batchهای آموزشی را پردازش می‌کند و زمان‌بر است. آن را یک‌بار اجرا کن؛ اجرای مجدد یک epoch آموزشی دیگر به همین مدل اضافه می‌کند، ولی برچسب `epoch=1` تاریخچه را از نو می‌نویسد.


In [ ]:
layer4_weights_before_epoch = fine_tune_model.layer4[0].conv1.weight.detach().clone()
bn_mean_before_epoch = fine_tune_model.layer4[0].bn1.running_mean.detach().clone()

fine_tune_train_metrics = train_fine_tune_one_epoch(
    fine_tune_model, resnet_train_loader, fine_tune_optimizer, loss_fn, resnet_device
)
fine_tune_val_metrics = evaluate_feature_head(
    fine_tune_model, resnet_val_loader, loss_fn, resnet_device
)
fine_tune_history = [{
    "epoch": 1,
    "train": fine_tune_train_metrics,
    "validation": fine_tune_val_metrics,
}]

layer4_changed = not torch.equal(
    layer4_weights_before_epoch, fine_tune_model.layer4[0].conv1.weight.detach()
)
bn_mean_unchanged = torch.equal(
    bn_mean_before_epoch, fine_tune_model.layer4[0].bn1.running_mean
)
assert layer4_changed
assert bn_mean_unchanged

print("Train loss:", round(fine_tune_train_metrics["loss"], 4))
print("Train accuracy:", f'{fine_tune_train_metrics["accuracy"]:.1%}')
print("Validation loss:", round(fine_tune_val_metrics["loss"], 4))
print("Validation accuracy:", f'{fine_tune_val_metrics["accuracy"]:.1%}')
print("Layer4 convolution weights changed:", layer4_changed)
print("BatchNorm running mean unchanged:", bn_mean_unchanged)


## ResNet18، قدم ۲۱: ادامهٔ fine-tuning تا تعداد epoch انتخاب‌شده

نتیجهٔ epoch اول در `fine_tune_history` وجود دارد. این سلول همان مدل و optimizer را از epoch دوم ادامه می‌دهد؛ بنابراین وزن‌ها از نو مقداردهی نمی‌شوند. پس از هر epoch، train و validation را ثبت و چاپ می‌کنیم.

طبق معیار اصلی فایل استاد، وضعیت دارای **کمترین validation loss** را با `deepcopy(state_dict())` نگه می‌داریم. چون در آزمایش قبلی بیشترین دقت با کمترین loss هم‌زمان نبود، وضعیت دارای **بیشترین validation accuracy** را هم جدا در حافظه نگه می‌داریم. در پایان، `fine_tune_model` را به بهترین وضعیت از نظر loss برمی‌گردانیم. کپیِ بهترین دقت در `best_fine_accuracy_state` باقی می‌ماند تا بعداً بتوانیم آن را جدا ارزیابی یا ذخیره کنیم؛ این سلول هنوز فایل checkpoint جدیدی نمی‌نویسد.

این سلول پس از epoch اول، تا مقدار `FINE_TUNE_EPOCHS` ادامه می‌دهد و زمان‌بر است. آن را فقط یک‌بار در ادامهٔ قدم ۲۰ اجرا کن. مقایسه با مدلِ فقط-`fc` اثر خالصِ بازکردن `layer4` را ثابت نمی‌کند، چون مرحلهٔ fine-tuning گام‌های آموزشی بیشتری هم دارد.

**اجرای دوباره با تعداد epoch متفاوت:** فقط تغییر `FINE_TUNE_EPOCHS` و اجرای دوبارهٔ این سلول، آموزش تازه نیست؛ مدل و optimizer قبلی را ادامه می‌دهد. برای اجرای مستقل ۳۰‌epochی، قدم‌های ۱۶ و ۱۷ را برای ساخت مدل و optimizer جدید، سپس قدم ۲۰ را برای epoch اول، و در پایان همین قدم ۲۱ را اجرا کن. این سلول اگر history از قبل بیش از یک epoch داشته باشد، متوقف می‌شود.


In [ ]:
from copy import deepcopy

# This cell continues the one-epoch run exactly once. Rebuild the model and
# optimizer in steps 16-17, then rerun step 20 before starting a new run.
if len(fine_tune_history) != 1 or fine_tune_history[0]["epoch"] != 1:
    raise RuntimeError(
        "This fine-tuning continuation has already run. "
        "Recreate the fine-tune model and optimizer (steps 16-17), "
        "then rerun the first epoch (step 20) before continuing."
    )

FINE_TUNE_EPOCHS = 30
best_fine_val_loss = fine_tune_val_metrics["loss"]
best_fine_loss_epoch = 1
best_fine_loss_state = deepcopy(fine_tune_model.state_dict())
best_fine_val_accuracy = fine_tune_val_metrics["accuracy"]
best_fine_accuracy_epoch = 1
best_fine_accuracy_state = deepcopy(fine_tune_model.state_dict())

print(
    f"Epoch 01/{FINE_TUNE_EPOCHS} | "
    f"train acc: {fine_tune_train_metrics['accuracy']:.1%} | "
    f"val acc: {fine_tune_val_metrics['accuracy']:.1%} | "
    f"val loss: {fine_tune_val_metrics['loss']:.4f}"
)

for epoch in range(2, FINE_TUNE_EPOCHS + 1):
    train_metrics = train_fine_tune_one_epoch(
        fine_tune_model, resnet_train_loader, fine_tune_optimizer, loss_fn, resnet_device
    )
    val_metrics = evaluate_feature_head(
        fine_tune_model, resnet_val_loader, loss_fn, resnet_device
    )
    fine_tune_history.append({
        "epoch": epoch,
        "train": train_metrics,
        "validation": val_metrics,
    })

    if val_metrics["loss"] < best_fine_val_loss:
        best_fine_val_loss = val_metrics["loss"]
        best_fine_loss_epoch = epoch
        best_fine_loss_state = deepcopy(fine_tune_model.state_dict())
    if val_metrics["accuracy"] > best_fine_val_accuracy:
        best_fine_val_accuracy = val_metrics["accuracy"]
        best_fine_accuracy_epoch = epoch
        best_fine_accuracy_state = deepcopy(fine_tune_model.state_dict())

    print(
        f"Epoch {epoch:02d}/{FINE_TUNE_EPOCHS} | "
        f"train acc: {train_metrics['accuracy']:.1%} | "
        f"val acc: {val_metrics['accuracy']:.1%} | "
        f"val loss: {val_metrics['loss']:.4f}"
    )

fine_tune_model.load_state_dict(best_fine_loss_state)
print("Best epoch by validation loss:", best_fine_loss_epoch)
print("Best validation loss:", round(best_fine_val_loss, 4))
print("Best epoch by validation accuracy:", best_fine_accuracy_epoch)
print("Best validation accuracy:", f'{best_fine_val_accuracy:.1%}')

# Keep the measured fine-tuning result tied to this split.
fine_result = {
    "experiment": "resnet18_layer4_finetune",
    "architecture": "resnet18", "training_mode": "layer4_plus_fc",
    "split_fingerprint": resnet_manifest["fingerprint"],
    "result_status": "current_split",
    "training_images": len(resnet_training_rows),
    "validation_images": len(resnet_validation_rows),
    "epochs": len(fine_tune_history),
    "selected_epoch": best_fine_loss_epoch,
    "selection_metric": "validation_loss",
    "selected_validation_loss": best_fine_val_loss,
    "selected_validation_accuracy": fine_tune_history[best_fine_loss_epoch - 1]["validation"]["accuracy"],
    "highest_validation_accuracy_epoch": best_fine_accuracy_epoch,
    "highest_validation_accuracy": best_fine_val_accuracy,
    "history": fine_tune_history,
}
results_path = project_root / "reports/experiment_results.json"
all_results = json.loads(results_path.read_text()) if results_path.exists() else {}
all_results["_split"] = {
    "current_fingerprint": resnet_manifest["fingerprint"],
    "comparison_rule": "Compare results only when split_fingerprint matches.",
}
all_results["resnet18_layer4_finetune"] = fine_result
results_path.write_text(json.dumps(all_results, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Recorded current fine-tuning result in:", results_path.name)
